## Solving GC_Kircher Header problem
- Header of GC_Kircher headers are too long and bwa-mem2 has problem with it.
- Add match the current header with a shorter header and add this to the reference fasta
- Notebook 198 (http://127.0.0.1:8888/tree?token=0436c0c7c46ad408178611d0238a86fce996eaf5f08d6361)

In [13]:
import pandas as pd
import os

In [18]:
def read_fasta(fasta_file):
    """
    Read fasta file and return a dataframe of header and sequence 
    """
    header = []
    sequences = []
    with open(fasta_file, "r") as handle:
        for line in handle:
            if line.startswith(">"):
                header.append(line.strip())
            else:
                sequences.append(line.strip())
    df = pd.DataFrame({"header": header, "sequence": sequences})
    return df

def write_fasta(df, output_file, columns=["header", "sequence"]):
    """
    Write a dataframe to a fasta file
    @param df: dataframe to write
    @param output_file: output file name
    @param columns: columns to write to file (expected two strings (first is header, second is sequence))
    """
    # only keep the columns we want
    df = df[columns]
    print(df.head())
    # write df as fasta to file
    with open(output_file, "w") as handle:
        for index, row in df.iterrows():
            handle.write(row[columns[0]] + "\n")
            handle.write(row[columns[1]] + "\n")
    return output_file


In [19]:
# /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/reference/reference_bwa-mem2.fa
# read the fasta file: each sequence one line
output_dir = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/reference/"
ref_fasta = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/reference/reference_bwa-mem2.fa"

ref_fasta_df = read_fasta(ref_fasta)
ref_fasta_df
# add different (shorter) identifier for each header of ">GC_Kircher:<shorter_identifier>" (hash of the header)
import hashlib
ref_fasta_df["hashs"] = ref_fasta_df["header"].apply(lambda x: hashlib.md5(x.encode()).hexdigest())
ref_fasta_df

# add new header column (new_header) with the normal header for all headers not starting with ">GC_Kircher:" and with ">GC_Kircher:<hashs>" for all headers starting with ">GC_Kircher:"
ref_fasta_df["new_header"] = ref_fasta_df["header"].apply(lambda x: x if not x.startswith(">GC_Kircher:") else ">GC_Kircher:" + hashlib.md5(x.encode()).hexdigest())

# duplicates in the hashs column?
# ref_fasta_df["hashs"].duplicated().sum() # No
# write the new fasta file using the new_header column
write_fasta(ref_fasta_df, os.path.join(output_dir, "reference_bwa-mem2_new_header.fa"), columns=["new_header", "sequence"])

                                          new_header  \
0  >cardiac_neuro_cava_random:SKI|ENSG00000157933...   
1  >cardiac_neuro_cava_random:SKI|ENSG00000157933...   
2  >cardiac_neuro_cava_random:SKI|ENSG00000157933...   
3  >cardiac_neuro_cava_random:SKI|ENSG00000157933...   
4  >cardiac_neuro_cava_random:SKI|ENSG00000157933...   

                                            sequence  
0  AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...  
1  AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...  
2  AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...  
3  AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...  
4  AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...  


'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/reference/reference_bwa-mem2_new_header.fa'